In [202]:
# Standard library
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Plots and statistics
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

# Machine learning
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LassoCV, RidgeCV
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

# Project module
import build_lagged_features as blf
from build_lagged_features import DATE, DEFAULT_LAGS, TARGET, build_lagged_frame, read_table

import io
from contextlib import redirect_stdout
from itertools import product

In [203]:
def find_project_root(start=None):
    """Walk upward from the notebook's folder until a 'data/' folder is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "data").is_dir():
            return folder
    raise FileNotFoundError(f"No 'data' folder found above {start}")

DATA_DIR = find_project_root() / "data"                        # the folder
DATA_FILE = DATA_DIR / "chlorophyll_student_2015_2023.xlsx"    # the input file
print(DATA_DIR)
print(DATA_FILE)

df = blf.read_table(DATA_FILE)

C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data
C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\chlorophyll_student_2015_2023.xlsx


In [204]:
# Plots and Data Profiling

# Visualize data
df.head()
print(df.describe())
# Lista de variáveis numéricas do dataset
num_vars = [
    'chlorophyll_a_mg_m3', 'sst_c', 'par_umol_m2_s', 'nitrate_umol_l',
    'wind_speed_m_s', 'upwelling_index', 'mixed_layer_depth_m', 'salinity_psu',
    'current_speed_m_s', 'river_discharge_index', 'cloud_fraction',
    'surface_pressure_hpa', 'turbidity_ntu'
]

# Configuração de estilo do Seaborn
sns.set_theme(style="whitegrid")

'''
-----------------------------------------------------------------------------------------------------------------------
Boxplots
'''

# Boxplots Individuais
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(num_vars):
    sns.boxplot(
        y=df[col], 
        ax=axes[i], 
        color='skyblue', 
        flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.6)
    )
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('')
    axes[i].set_xlabel('')

# Remover os subplots não utilizados na grelha 4x4 (temos 13 variáveis)
for j in range(len(num_vars), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Boxplots per Variable', fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig("images/boxplots")
plt.close()


# Boxplot Conjunto Normalizado (Z-Score)
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[num_vars].dropna()), columns=num_vars)

plt.figure(figsize=(16, 8))
sns.boxplot(
    data=df_scaled, 
    orient='v', 
    palette='Set3',
    flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5)
)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.ylabel('Deviation', fontsize=12)
plt.title('Normalized Boxplots per Variable (Z-Score)', fontsize=15, fontweight='bold')
plt.axhline(y=3, color='r', linestyle='--', alpha=0.7, label='Limit Z = +3')
plt.axhline(y=-3, color='r', linestyle='--', alpha=0.7, label='Limit Z = -3')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig("images/boxplots_normalizados")
plt.close()


'''
-----------------------------------------------------------------------------------------------------------------------
Results and Codification
'''

print("--- Resumo da Estrutura dos Dados ---")
print(f"Número de registos: {len(df)}")
print(f"Intervalo de datas: {df['date'].min().date()} até {df['date'].max().date()}")
print(f"Valores em falta por coluna:\n{df.isnull().sum()}\n")

# Codificação Sazonal
day_of_year = df['date'].dt.dayofyear
df['s_year'] = np.sin(2 * np.pi * day_of_year / 365.25)
df['c_year'] = np.cos(2 * np.pi * day_of_year / 365.25)


'''
-----------------------------------------------------------------------------------------------------------------------
Analysis of Clorofila-a 
'''

# Clorofila-a - Analise da Variável (line plot)
plt.figure(figsize=(14, 5))
plt.plot(df['date'], df['chlorophyll_a_mg_m3'], color='forestgreen', linewidth=1)
plt.title('Temporal Trend Plot for Clorofila-a (2015–2023)')
plt.xlabel('Data')
plt.ylabel('Chlorophyll-a (mg/m³)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("images/Clorofila-a_Temporal_Trend_Plot")
plt.close()


# Clorofila-a - Histogram
plt.figure(figsize=(10, 5))
plt.hist(
    df['chlorophyll_a_mg_m3'].dropna(), 
    bins=70, 
    color='forestgreen', 
    edgecolor='black', 
    alpha=0.7
)
plt.title('Histogram of Clorofila-a')
plt.xlabel('Chlorophyll-a (mg/m³)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("images/Clorofila-a_histograma")
plt.close()


# Clorofila-a - Histogram with better fit distributions
# Prepare the data
data = df['chlorophyll_a_mg_m3'].dropna()
plt.figure(figsize=(10, 5))
# Plot histogram with density=True so the total area equals 1 (matching the PDF curves)
count, bins, ignored = plt.hist(
    data, 
    bins=50, 
    density=True, 
    color='forestgreen', 
    edgecolor='black', 
    alpha=0.5,
    label=' Observed Data (Histogram)'
)
# Generate a smooth range of x values for the fitted curves
x = np.linspace(data.min(), data.max(), 200)
# Fit and plot Lognormal distribution
shape_ln, loc_ln, scale_ln = stats.lognorm.fit(data, floc=0)
pdf_lognorm = stats.lognorm.pdf(x, shape_ln, loc_ln, scale_ln)
plt.plot(x, pdf_lognorm, 'r-', linewidth=2, label='Lognormal Fit')
# Fit and plot Gamma distribution
shape_g, loc_g, scale_g = stats.gamma.fit(data, floc=0)
pdf_gamma = stats.gamma.pdf(x, shape_g, loc_g, scale_g)
plt.plot(x, pdf_gamma, 'b--', linewidth=2, label='Gamma Fit')
plt.title('Histogram for Clorofila-a with Distributions')
plt.xlabel('Chlorophyll-a (mg/m³)')
plt.ylabel('Frequancy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("images/Clorofila-a_histograma_distributions")
plt.close()


'''
-----------------------------------------------------------------------------------------------------------------------
Histograms
'''

# Histograms of all variables
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(num_vars):
    sns.histplot(
        df[col].dropna(), 
        ax=axes[i], 
        color='forestgreen', 
        bins=30, 
        edgecolor='black',
        alpha=0.6
    )
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('')
    axes[i].set_xlabel('')
# Remove unused subplots in the 4x4 grid
for j in range(len(num_vars), len(axes)):
    fig.delaxes(axes[j])
plt.suptitle('Histograms per Variable', fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig("images/histograms")
plt.close()


# Histograms of all variables with best fit distributions
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(num_vars):
    data = df[col].dropna()
    # Plot density histogram
    axes[i].hist(data, bins=30, density=True, color='forestgreen', edgecolor='black', alpha=0.4)
    # Generate smooth curve points
    xmin, xmax = axes[i].get_xlim()
    x = np.linspace(xmin, xmax, 100)
    try:
        # Fit Lognormal if strictly positive, else Normal
        if (data > 0).all():
            shape, loc, scale = stats.lognorm.fit(data, floc=0)
            p = stats.lognorm.pdf(x, shape, loc, scale)
        else:
            mu, std = stats.norm.fit(data)
            p = stats.norm.pdf(x, mu, std)
        axes[i].plot(x, p, 'r-', linewidth=2)
    except Exception:
        pass  # Skip if fitting fails for a specific edge-case column
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('')
    axes[i].set_xlabel('')
for j in range(len(num_vars), len(axes)):
    fig.delaxes(axes[j])
plt.suptitle('Histograms with Best-Fit Distributions per Variable', fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig("images/histograms_distributions")
plt.close()


'''
-----------------------------------------------------------------------------------------------------------------------
Estacionariedade
'''

# Teste de Estacionariedade (ADF Test)
adf_result = adfuller(df['chlorophyll_a_mg_m3'].dropna(), result_object=True)
print(f"Estatística ADF: {adf_result[0]:.4f}")
print(f"p-value ADF: {adf_result[1]:.4e}")


'''
-----------------------------------------------------------------------------------------------------------------------
Correlation matrix and Scatter plot
'''

# Matriz de Correlacao entre Variaveis Exógenas e Alvo
exogenous_cols = [
    'sst_c', 'par_umol_m2_s', 'nitrate_umol_l', 'wind_speed_m_s',
    'upwelling_index', 'mixed_layer_depth_m', 'salinity_psu',
    'current_speed_m_s', 'river_discharge_index', 'cloud_fraction',
    'surface_pressure_hpa', 'turbidity_ntu'
]
corr_matrix = df[['chlorophyll_a_mg_m3'] + exogenous_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlação de Pearson')
plt.tight_layout()
plt.savefig("images/matrix_correlacao")
plt.close()


# Pairplot matrix for all numerical variables
pairplot_fig = sns.pairplot(
    df[num_vars].dropna(), 
    diag_kind='kde', 
    plot_kws={'alpha': 0.4, 's': 10, 'color': 'forestgreen'},
    diag_kws={'color': 'forestgreen'}
)
pairplot_fig.fig.suptitle('Scatter Plot Matrix', y=1.02, fontsize=16, fontweight='bold')
pairplot_fig.savefig("images/scatterplot.png", dpi=300)
plt.close()


'''
-----------------------------------------------------------------------------------------------------------------------
Correlation with Lags
'''


# Correlacao Cruzada com Lags das Exogenas (para selecionar lags no ARX)
lags = range(10, 15)
target = df['chlorophyll_a_mg_m3']

lag_corrs = pd.DataFrame(index=exogenous_cols, columns=[f'lag_{k}' for k in lags])

for col in exogenous_cols:
    for k in lags:
        lag_corrs.loc[col, f'lag_{k}'] = target.corr(df[col].shift(k))

plt.figure(figsize=(10, 6))
sns.heatmap(lag_corrs.astype(float), annot=True, fmt=".2f", cmap='viridis')
plt.title('Correlação Cruzada com Clorofila-a $y(t)$')
plt.xlabel('Atraso (Lags em dias)')
plt.tight_layout()
plt.savefig("images/correlacao_cruzada_lags")
plt.close()


                      date  chlorophyll_a_mg_m3        sst_c  par_umol_m2_s  \
count                 3287          3287.000000  3287.000000    3287.000000   
mean   2019-07-02 00:00:00             1.747175    15.302841     310.625146   
min    2015-01-01 00:00:00             0.276926     9.228039       8.000000   
25%    2017-04-01 12:00:00             1.312216    12.121438     149.255064   
50%    2019-07-02 00:00:00             1.624492    15.420263     311.322698   
75%    2021-09-30 12:00:00             2.074589    18.478341     470.651450   
max    2023-12-31 00:00:00            10.811834    21.275018     665.151866   
std                    NaN             0.633770     3.266127     169.246168   

       nitrate_umol_l  wind_speed_m_s  upwelling_index  mixed_layer_depth_m  \
count     3287.000000     3287.000000      3287.000000          3287.000000   
mean         4.402534        5.976806        -0.026088            23.929379   
min          0.811167        1.457805        -3.172

In [205]:
def make_lags(**overrides):
    """Copy of the default lags with changes.
    int N -> lags 0..N (1..N for the target); list -> used as is; None -> drop the variable."""
    spec = {k: list(v) for k, v in blf.DEFAULT_LAGS.items()}
    for col, val in overrides.items():
        if val is None:
            spec.pop(col, None)
        elif isinstance(val, int):
            spec[col] = list(range(1 if col == blf.TARGET else 0, val + 1))
        else:
            spec[col] = list(val)
    return spec

def build_and_save(raw, name, lag_spec=None, fmt="csv"):
    """Build the lagged frame and save it as <project>/data/<name>.<fmt>."""
    lagged = blf.build_lagged_frame(raw, lag_spec=lag_spec)
    path = DATA_DIR / f"{name}.{fmt}"
    if fmt == "xlsx":
        lagged.to_excel(path, sheet_name="lagged_data", index=False)
    else:
        lagged.to_csv(path, index=False)
    print(f"saved {lagged.shape} to {path}")
    return lagged

In [206]:
SCALERS = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}

# helpers


def make_masks(dates, split):
    """split can be:
       '2023-01-01'                       -> train < date, test >= date
       ('2022-01-01', '2024-01-01')       -> test = [start, end), train = everything else
       ('2022-01-01', '2023-01-01', 'past') -> test = [start, end), train = only dates before start
    """
    if isinstance(split, (tuple, list)):
        start, end = pd.Timestamp(split[0]), pd.Timestamp(split[1])
        test = (dates >= start) & (dates < end)
        train = (dates < start) if (len(split) > 2 and split[2] == "past") else ~test
    else:
        train = dates < pd.Timestamp(split)
        test = ~train
    return train, test


def scale_sets(scaling, X_fit, *others):
    """Fit the scaler on X_fit only and apply it to the other sets."""
    if not scaling:
        return (X_fit, *others)
    scaler = SCALERS["standard" if scaling is True else scaling]()
    out = [pd.DataFrame(scaler.fit_transform(X_fit), columns=X_fit.columns, index=X_fit.index)]
    out += [pd.DataFrame(scaler.transform(o), columns=o.columns, index=o.index) for o in others]
    return tuple(out)


def year_folds(dates, fold_years=1, scheme="blocked"):
    """Cross-validation folds made of whole years. `dates` = Series of dates (index = row labels).
    fold_years=1 -> one fold per year; fold_years=3 -> one fold per 3 years.
    scheme='blocked': validate on a block, train on all the other years.
    scheme='forward': validate on a block, train only on the years before it (respects time)."""
    years = dates.dt.year
    uniq = sorted(years.unique())
    blocks = [uniq[i:i + fold_years] for i in range(0, len(uniq), fold_years)]
    folds = []
    for k, b in enumerate(blocks):
        va = years.isin(b)
        if scheme == "blocked":
            tr = ~va
        elif scheme == "forward":
            if k == 0:
                continue
            tr = years < b[0]
        else:
            raise ValueError("scheme must be 'blocked' or 'forward'")
        folds.append((dates.index[tr], dates.index[va]))
    return folds


def prepare_data(lagged_data, scaling, split):
    """Split (see make_masks) + optional scaling (scaler fit on train only)."""
    n_before = len(lagged_data)
    lagged_data = lagged_data.dropna()
    if len(lagged_data) < n_before:
        print(f"Dropped {n_before - len(lagged_data)} rows with NaN (from removed outliers)")

    feature_cols = [c for c in lagged_data.columns if c not in [blf.DATE, blf.TARGET]]
    X, y = lagged_data[feature_cols], lagged_data[blf.TARGET]
    train_m, test_m = make_masks(lagged_data[blf.DATE], split)
    X_train, X_test = scale_sets(scaling, X[train_m], X[test_m])
    return X_train, X_test, y[train_m], y[test_m]


def compare_splits(raw, model_cls, splits, params=None, scaling=None,
                   outlier=None, lag_spec=None):
    """Evaluate ONE model setup on several train/test splits.
    splits : {"name": split_spec}, see make_masks
    outlier: None or dict(threshold=6, method='months_back', months=12)"""
    params, rows = params or {}, []
    with redirect_stdout(io.StringIO()):
        for name, sp in splits.items():
            clean = raw if outlier is None else handle_outliers(raw, blf.TARGET, fit_until=sp, **outlier)
            lagged = blf.build_lagged_frame(clean, lag_spec=lag_spec)
            X_tr, X_te, y_tr, y_te = prepare_data(lagged, scaling, sp)
            pred = model_cls(**params).fit(X_tr, y_tr).predict(X_te)
            rows.append({"split": name, "n_train": len(X_tr), "n_test": len(X_te),
                         **calc_metrics(y_te, pred)})
    return pd.DataFrame(rows).set_index("split")






'''
def prepare_data(lagged_data, scaling, split):
    """Chronological split + optional scaling (scaler fit on train only).
    scaling: None/False, True or 'standard', 'minmax', 'robust'."""

    # VER MELHOR OQ EQ ISTO FAZ PQ ISTO TIRA ALGUMAS COLUNAS POR CAUSA DO REMOVE OUTLIERS
    n_before = len(lagged_data)
    lagged_data = lagged_data.dropna()
    if len(lagged_data) < n_before:
        print(f"Dropped {n_before - len(lagged_data)} rows with NaN (from removed outliers)")


    date_col, target_col = blf.DATE, blf.TARGET
    feature_cols = [c for c in lagged_data.columns if c not in [date_col, target_col]]
    X, y = lagged_data[feature_cols], lagged_data[target_col]

    train_mask = lagged_data[date_col] < split
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[~train_mask], y[~train_mask]

    if scaling:
        name = "standard" if scaling is True else scaling
        scaler = SCALERS[name]()
        X_train = pd.DataFrame(scaler.fit_transform(X_train),
                               columns=X_train.columns, index=X_train.index)
        X_test = pd.DataFrame(scaler.transform(X_test),
                              columns=X_test.columns, index=X_test.index)
    return X_train, X_test, y_train, y_test

'''


def report_metrics(y_true, y_pred, label="X_test"):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"--- Metrics ({label}) ---\n"
          f"R2: {r2}\n"
          f"MAE: {mae}\n"
          f"MSE: {mse}\n"
          f"RMSE: {rmse}\n"
          f"MAPE: {mape}")


def report_ols_summary(model_sm, sig):
    """statsmodels OLS: header table, all coefficients, significant ones."""
    print(model_sm.summary().tables[0])
    coefs = model_sm.summary2().tables[1]
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
        display(coefs)
    display(coefs[coefs["P>|t|"] < sig].sort_values("P>|t|"))


def report_coefs(model, columns, nonzero_only=False):
    """sklearn linear models: intercept + coefficients sorted by absolute value."""
    coefs = pd.DataFrame({"coef": model.coef_}, index=columns)
    coefs["abs_coef"] = coefs["coef"].abs()
    coefs = coefs.sort_values("abs_coef", ascending=False)
    print(f"Intercept: {float(model.intercept_)}")
    print(f"Non-zero coefficients: {(coefs['coef'] != 0).sum()} of {len(coefs)}")
    if nonzero_only:
        coefs = coefs[coefs["coef"] != 0]
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
        display(coefs)
    return coefs






def handle_outliers(df, col, threshold, method="median", months=12,
                    lower=None, fit_until=None):
    """Treat values of `col` above `threshold` (and below `lower`, if given) as outliers.

    method:
      'remove'      -> set to NaN (rows are kept, so the daily lags stay aligned)
      'neighbors'   -> replace with the mean of the day before and the day after;
                       if only one of them is valid, use it; if neither, use the median
      'mean'        -> replace with the mean of the normal values
      'median'      -> replace with the median of the normal values
      'mode'        -> replace with the mode (values rounded to 2 decimals first)
      'months_back' -> replace with the value from `months` months earlier;
                       if that value is missing or also an outlier, use the median
      'years_median' -> replace with the median of the same calendar day in all
                        previous years (outliers excluded); if there is none, use the median
       fit_until: date (e.g. '2022-01-01'). Mean/median/mode are computed only from
               data before this date, so nothing from the test period leaks in.
    """
    out_df = df.copy()
    s = out_df[col].astype(float)

    is_out = s > threshold
    if lower is not None:
        is_out |= s < lower

    # reference values for the statistics: normal values, train period only

    ref = ~is_out
    if fit_until is not None:
        ref &= make_masks(out_df[blf.DATE], fit_until)[0]

    '''
    ref = ~is_out
    if fit_until is not None:
        ref &= out_df[blf.DATE] < pd.Timestamp(fit_until)
    '''
    
    ref_vals = s[ref]
    median = ref_vals.median()

    if method == "remove":
        s[is_out] = np.nan
    elif method == "neighbors":
        clean = s.where(~is_out)                      # outliers = NaN
        pair = pd.concat([clean.shift(1), clean.shift(-1)], axis=1)
        neighbor_mean = pair.mean(axis=1)             # mean of day before and day after
        s[is_out] = neighbor_mean[is_out].fillna(median)
    elif method == "mean":
        s[is_out] = ref_vals.mean()
    elif method == "median":
        s[is_out] = median
    elif method == "mode":
        s[is_out] = ref_vals.round(2).mode().iloc[0]
    elif method == "months_back":
        clean = pd.Series(s.where(~is_out).values, index=out_df[blf.DATE])  # outliers = NaN
        for i in s.index[is_out]:
            past = out_df.at[i, blf.DATE] - pd.DateOffset(months=months)
            value = clean.get(past, np.nan)
            s[i] = value if pd.notna(value) else median
    elif method == "years_median":
        clean = pd.Series(s.where(~is_out).values, index=out_df[blf.DATE])  # outliers = NaN
        first_year = clean.index.min().year
        for i in s.index[is_out]:
            day = out_df.at[i, blf.DATE]
            past = [clean.get(day - pd.DateOffset(years=k), np.nan)
                    for k in range(1, day.year - first_year + 1)]
            value = np.nanmedian(past) if len(past) and not np.all(np.isnan(past)) else np.nan
            s[i] = value if pd.notna(value) else median
    else:
        raise ValueError("method must be remove, neighbors, mean, median, mode, months_back or years_median")
    
    out_df[col] = s
    print(f"{col}: {int(is_out.sum())} outliers (> {threshold}) handled with '{method}'")
    return out_df


# ---------- models ----------
def analysis_LOS(lagged_data, scaling, sig, split):
    """Linear regression. `sig` = significance level for the p-values."""
    X_train, X_test, y_train, y_test = prepare_data(lagged_data, scaling, split)
    model1 = LinearRegression(fit_intercept=True).fit(X_train, y_train)
    report_metrics(y_test, model1.predict(X_test))

    model2 = sm.OLS(y_train, sm.add_constant(X_train)).fit()
    report_ols_summary(model2, sig)
    return model1, model2


def analysis_lasso(lagged_data, scaling, alpha, split, max_iter=10000):
    if not scaling:
        raise ValueError("Lasso need scaling: use 'standard', 'minmax' or 'robust'")
    """Lasso. `alpha` = penalty strength (higher = more coefficients set to 0)."""
    X_train, X_test, y_train, y_test = prepare_data(lagged_data, scaling, split)
    model = Lasso(alpha=alpha, max_iter=max_iter).fit(X_train, y_train)
    report_metrics(y_test, model.predict(X_test))
    report_coefs(model, X_train.columns, nonzero_only=True)
    return model


def analysis_ridge(lagged_data, scaling, alpha, split):
    if not scaling:
        raise ValueError("Ridge need scaling: use 'standard', 'minmax' or 'robust'")
    """Ridge. `alpha` = penalty strength (higher = smaller coefficients)."""
    X_train, X_test, y_train, y_test = prepare_data(lagged_data, scaling, split)
    model = Ridge(alpha=alpha).fit(X_train, y_train)
    report_metrics(y_test, model.predict(X_test))
    report_coefs(model, X_train.columns)
    return model

In [207]:
# m1, m2 = analysis_LOS(lagged, scaling="standard", sig=0.05, split="2022-01-01")
# lasso  = analysis_lasso(lagged, scaling="standard", alpha=0.01, split="2022-01-01")
# ridge  = analysis_ridge(lagged, scaling="standard", alpha=10, split="2022-01-01")

'''
# pick the threshold by looking at the histogram, or use a rule such as:
q1, q3 = df[blf.TARGET].quantile([0.25, 0.75])
print("Q3 + 3*IQR =", q3 + 3 * (q3 - q1))
'''


'\n# pick the threshold by looking at the histogram, or use a rule such as:\nq1, q3 = df[blf.TARGET].quantile([0.25, 0.75])\nprint("Q3 + 3*IQR =", q3 + 3 * (q3 - q1))\n'

In [208]:
# Sem Mudancas | Lag Default | No Scaling | sig=0.05
default_lagged = build_and_save(df, "lagged_default")
analysis_LOS(default_lagged, scaling=None, sig=0.05, split="2023-01-01")

saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
--- Metrics (X_test) ---
R2: 0.5035694401864301
MAE: 0.22434881165556464
MSE: 0.29931942590495564
RMSE: 0.5471009284446112
MAPE: 0.11839472434277365
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.758
Model:                             OLS   Adj. R-squared:                  0.747
Method:                  Least Squares   F-statistic:                     65.41
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:27   Log-Likelihood:                -638.13
No. Observations:                 2908   AIC:                             1544.
Df Residuals:                     2774   BIC:                             2345.
Df Model:                          133                                         
Covariance Type:            

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,5.475371,9.524580,0.574867,5.654276e-01,-13.200612,24.151353
chlorophyll_a_mg_m3__lag_1,0.383118,0.018960,20.206216,8.195510e-85,0.345940,0.420296
chlorophyll_a_mg_m3__lag_2,0.030519,0.020316,1.502206,1.331578e-01,-0.009317,0.070356
chlorophyll_a_mg_m3__lag_3,0.009831,0.020331,0.483572,6.287282e-01,-0.030034,0.049697
chlorophyll_a_mg_m3__lag_4,-0.008582,0.020371,-0.421280,6.735835e-01,-0.048526,0.031362
chlorophyll_a_mg_m3__lag_5,-0.011474,0.020320,-0.564686,5.723332e-01,-0.051319,0.028370
chlorophyll_a_mg_m3__lag_6,0.028817,0.020307,1.419064,1.559928e-01,-0.011002,0.068636
chlorophyll_a_mg_m3__lag_7,0.046118,0.020131,2.290955,2.204068e-02,0.006646,0.085590
chlorophyll_a_mg_m3__lag_8,0.057973,0.020107,2.883204,3.966820e-03,0.018547,0.097399
chlorophyll_a_mg_m3__lag_9,0.008043,0.020101,0.400127,6.890941e-01,-0.031372,0.047459


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.383118,0.018960,20.206216,8.195510e-85,0.345940,0.420296
wind_speed_m_s__lag_1,0.068624,0.012338,5.561769,2.925163e-08,0.044430,0.092817
nitrate_umol_l__lag_3,0.123845,0.024290,5.098664,3.650758e-07,0.076218,0.171473
par_umol_m2_s__lag_1,0.000885,0.000185,4.787904,1.773769e-06,0.000522,0.001247
upwelling_index__lag_2,0.109604,0.023882,4.589309,4.644842e-06,0.062775,0.156433
mixed_layer_depth_m__lag_1,-0.011689,0.003479,-3.359692,7.908457e-04,-0.018511,-0.004867
nitrate_umol_l__lag_4,0.073519,0.024427,3.009693,2.638664e-03,0.025621,0.121417
chlorophyll_a_mg_m3__lag_8,0.057973,0.020107,2.883204,3.966820e-03,0.018547,0.097399
chlorophyll_a_mg_m3__lag_7,0.046118,0.020131,2.290955,2.204068e-02,0.006646,0.085590
surface_pressure_hpa__lag_1,-0.009385,0.004292,-2.186331,2.887469e-02,-0.017802,-0.000968


(LinearRegression(),
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x2c5aafe3ad0>)

In [209]:
# Outliers - Remove | Lag Default | No Scaling | sig=0.05
df_outliers_remove = handle_outliers(df, blf.TARGET, threshold=6,
                            method="remove", months=12, fit_until="2023-01-01")

remove_lagged = build_and_save(df_outliers_remove, "lagged_default")
analysis_LOS(remove_lagged, scaling=None, sig=0.05, split="2023-01-01")

chlorophyll_a_mg_m3: 4 outliers (> 6) handled with 'remove'
saved (3269, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
Dropped 56 rows with NaN (from removed outliers)
--- Metrics (X_test) ---
R2: 0.733739492812836
MAE: 0.18524378280820286
MSE: 0.095973439446389
RMSE: 0.30979580282242203
MAPE: 0.10955119451402122
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.834
Model:                             OLS   Adj. R-squared:                  0.825
Method:                  Least Squares   F-statistic:                     102.7
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:27   Log-Likelihood:                 63.909
No. Observations:                 2863   AIC:                             140.2
Df Residuals:                     2729   BIC:                             938.8
D

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,2.193343,7.556783,0.290248,7.716483e-01,-12.624251,17.010938
chlorophyll_a_mg_m3__lag_1,0.560357,0.019156,29.252582,7.382055e-164,0.522795,0.597918
chlorophyll_a_mg_m3__lag_2,-0.036992,0.021967,-1.683947,9.230625e-02,-0.080066,0.006082
chlorophyll_a_mg_m3__lag_3,-0.016880,0.022002,-0.767219,4.430176e-01,-0.060021,0.026261
chlorophyll_a_mg_m3__lag_4,0.018301,0.022026,0.830876,4.061161e-01,-0.024888,0.061489
chlorophyll_a_mg_m3__lag_5,-0.018304,0.021934,-0.834481,4.040831e-01,-0.061314,0.024706
chlorophyll_a_mg_m3__lag_6,0.036829,0.021931,1.679308,9.320658e-02,-0.006174,0.079833
chlorophyll_a_mg_m3__lag_7,0.056851,0.021635,2.627713,8.644247e-03,0.014428,0.099273
chlorophyll_a_mg_m3__lag_8,0.041096,0.021581,1.904301,5.697610e-02,-0.001220,0.083413
chlorophyll_a_mg_m3__lag_9,0.017104,0.021507,0.795296,4.265103e-01,-0.025067,0.059275


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.560357,0.019156,29.252582,7.382055e-164,0.522795,0.597918
wind_speed_m_s__lag_1,0.074768,0.009769,7.653458,2.695441e-14,0.055612,0.093924
upwelling_index__lag_2,0.130585,0.018887,6.913836,5.851661e-12,0.093550,0.167620
par_umol_m2_s__lag_1,0.000923,0.000146,6.304063,3.370160e-10,0.000636,0.001210
nitrate_umol_l__lag_3,0.118186,0.019229,6.146308,9.089177e-10,0.080481,0.155890
mixed_layer_depth_m__lag_1,-0.011448,0.002754,-4.157153,3.321641e-05,-0.016848,-0.006048
sst_c__lag_5,0.045807,0.017166,2.668543,7.663176e-03,0.012148,0.079467
chlorophyll_a_mg_m3__lag_7,0.056851,0.021635,2.627713,8.644247e-03,0.014428,0.099273
turbidity_ntu__lag_4,-0.056221,0.021570,-2.606404,9.199617e-03,-0.098516,-0.013925


(LinearRegression(),
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x2c5afba15b0>)

In [210]:
# Outliers - Mean | Lag Default | No Scaling | sig=0.05
df_outliers_mean = handle_outliers(df, blf.TARGET, threshold=6,
                            method="mean", months=12, fit_until="2023-01-01")

mean_lagged = build_and_save(df_outliers_mean, "lagged_default")
analysis_LOS(mean_lagged, scaling=None, sig=0.05, split="2023-01-01")

chlorophyll_a_mg_m3: 4 outliers (> 6) handled with 'mean'
saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
--- Metrics (X_test) ---
R2: 0.7451672784847836
MAE: 0.1875484823042481
MSE: 0.0972906136074033
RMSE: 0.31191443315018835
MAPE: 0.10919935654719734
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.833
Model:                             OLS   Adj. R-squared:                  0.825
Method:                  Least Squares   F-statistic:                     103.9
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:28   Log-Likelihood:                 50.197
No. Observations:                 2908   AIC:                             167.6
Df Residuals:                     2774   BIC:                             968.3
Df Model:                          133             

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,3.106449,7.511201,0.413576,6.792170e-01,-11.621661,17.834560
chlorophyll_a_mg_m3__lag_1,0.560511,0.018994,29.509530,1.086201e-166,0.523267,0.597755
chlorophyll_a_mg_m3__lag_2,-0.031551,0.021788,-1.448071,1.477103e-01,-0.074273,0.011172
chlorophyll_a_mg_m3__lag_3,-0.023255,0.021793,-1.067067,2.860344e-01,-0.065987,0.019478
chlorophyll_a_mg_m3__lag_4,0.020379,0.021803,0.934706,3.500212e-01,-0.022372,0.063130
chlorophyll_a_mg_m3__lag_5,-0.020982,0.021724,-0.965854,3.342017e-01,-0.063578,0.021614
chlorophyll_a_mg_m3__lag_6,0.036100,0.021720,1.662101,9.660547e-02,-0.006488,0.078689
chlorophyll_a_mg_m3__lag_7,0.061264,0.021430,2.858819,4.283948e-03,0.019244,0.103285
chlorophyll_a_mg_m3__lag_8,0.032777,0.021382,1.532904,1.254135e-01,-0.009150,0.074703
chlorophyll_a_mg_m3__lag_9,0.023795,0.021311,1.116555,2.642813e-01,-0.017993,0.065583


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.560511,0.018994,29.509530,1.086201e-166,0.523267,0.597755
wind_speed_m_s__lag_1,0.075771,0.009737,7.782072,9.988895e-15,0.056679,0.094863
upwelling_index__lag_2,0.132807,0.018833,7.051903,2.219161e-12,0.095879,0.169734
par_umol_m2_s__lag_1,0.000941,0.000146,6.451076,1.305777e-10,0.000655,0.001227
nitrate_umol_l__lag_3,0.115203,0.019165,6.011233,2.082285e-09,0.077625,0.152782
mixed_layer_depth_m__lag_1,-0.011428,0.002750,-4.155095,3.349851e-05,-0.016820,-0.006035
chlorophyll_a_mg_m3__lag_7,0.061264,0.021430,2.858819,4.283948e-03,0.019244,0.103285
sst_c__lag_5,0.044091,0.017109,2.577104,1.001440e-02,0.010544,0.077638
turbidity_ntu__lag_4,-0.054727,0.021503,-2.545035,1.098041e-02,-0.096892,-0.012563
nitrate_umol_l__lag_13,-0.035296,0.016425,-2.148880,3.173019e-02,-0.067502,-0.003089


(LinearRegression(),
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x2c5ab3623f0>)

In [211]:
# Outliers - Mediana | Lag Default | No Scaling | sig=0.05
df_outliers_median = handle_outliers(df, blf.TARGET, threshold=6,
                            method="median", months=12, fit_until="2023-01-01")

median_lagged = build_and_save(df_outliers_median, "lagged_default")
analysis_LOS(median_lagged, scaling=None, sig=0.05, split="2023-01-01")

chlorophyll_a_mg_m3: 4 outliers (> 6) handled with 'median'
saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
--- Metrics (X_test) ---
R2: 0.74278817716561
MAE: 0.1881878778081586
MSE: 0.09822200397059888
RMSE: 0.31340389909922767
MAPE: 0.10965092835939344
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.832
Model:                             OLS   Adj. R-squared:                  0.824
Method:                  Least Squares   F-statistic:                     103.5
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:28   Log-Likelihood:                 44.912
No. Observations:                 2908   AIC:                             178.2
Df Residuals:                     2774   BIC:                             978.9
Df Model:                          133            

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,3.112880,7.524777,0.413684,6.791375e-01,-11.641850,17.867609
chlorophyll_a_mg_m3__lag_1,0.558933,0.018995,29.425808,7.126178e-166,0.521687,0.596178
chlorophyll_a_mg_m3__lag_2,-0.029994,0.021773,-1.377584,1.684430e-01,-0.072687,0.012699
chlorophyll_a_mg_m3__lag_3,-0.024133,0.021777,-1.108189,2.678761e-01,-0.066833,0.018568
chlorophyll_a_mg_m3__lag_4,0.020920,0.021786,0.960242,3.370173e-01,-0.021798,0.063637
chlorophyll_a_mg_m3__lag_5,-0.020873,0.021707,-0.961589,3.363398e-01,-0.063437,0.021690
chlorophyll_a_mg_m3__lag_6,0.035790,0.021703,1.649107,9.923885e-02,-0.006765,0.078345
chlorophyll_a_mg_m3__lag_7,0.061714,0.021413,2.882129,3.980347e-03,0.019728,0.103701
chlorophyll_a_mg_m3__lag_8,0.032143,0.021365,1.504457,1.325775e-01,-0.009750,0.074036
chlorophyll_a_mg_m3__lag_9,0.024662,0.021294,1.158169,2.468948e-01,-0.017092,0.066416


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.558933,0.018995,29.425808,7.126178e-166,0.521687,0.596178
wind_speed_m_s__lag_1,0.075981,0.009754,7.789357,9.441186e-15,0.056854,0.095108
upwelling_index__lag_2,0.133332,0.018867,7.066959,1.995113e-12,0.096337,0.170326
par_umol_m2_s__lag_1,0.000942,0.000146,6.447329,1.337974e-10,0.000655,0.001228
nitrate_umol_l__lag_3,0.115058,0.019199,5.992821,2.328991e-09,0.077412,0.152705
mixed_layer_depth_m__lag_1,-0.011416,0.002755,-4.143270,3.526304e-05,-0.016819,-0.006013
chlorophyll_a_mg_m3__lag_7,0.061714,0.021413,2.882129,3.980347e-03,0.019728,0.103701
sst_c__lag_5,0.044206,0.017139,2.579248,9.952610e-03,0.010599,0.077812
turbidity_ntu__lag_4,-0.055331,0.021543,-2.568415,1.026835e-02,-0.097572,-0.013089
nitrate_umol_l__lag_13,-0.035664,0.016455,-2.167390,3.028993e-02,-0.067929,-0.003399


(LinearRegression(),
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x2c5b98eb710>)

In [212]:
# Outliers - Moda | Lag Default | No Scaling | sig=0.05
df_outliers_mode = handle_outliers(df, blf.TARGET, threshold=6,
                            method="mode", months=12, fit_until="2023-01-01")

mode_lagged = build_and_save(df_outliers_mode, "lagged_default")
analysis_LOS(mode_lagged, scaling=None, sig=0.05, split="2023-01-01")

chlorophyll_a_mg_m3: 4 outliers (> 6) handled with 'mode'
saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
--- Metrics (X_test) ---
R2: 0.7371568470277629
MAE: 0.1895691292281466
MSE: 0.10048556483870894
RMSE: 0.3169945817182195
MAPE: 0.11079717960500236
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.831
Model:                             OLS   Adj. R-squared:                  0.823
Method:                  Least Squares   F-statistic:                     102.3
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:28   Log-Likelihood:                 30.894
No. Observations:                 2908   AIC:                             206.2
Df Residuals:                     2774   BIC:                             1007.
Df Model:                          133             

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,3.130813,7.560961,0.414076,6.788504e-01,-11.694868,17.956493
chlorophyll_a_mg_m3__lag_1,0.554768,0.018995,29.205352,9.958977e-164,0.517522,0.592015
chlorophyll_a_mg_m3__lag_2,-0.026250,0.021734,-1.207777,2.272360e-01,-0.068867,0.016367
chlorophyll_a_mg_m3__lag_3,-0.025931,0.021734,-1.193090,2.329362e-01,-0.068548,0.016686
chlorophyll_a_mg_m3__lag_4,0.021929,0.021742,1.008604,3.132524e-01,-0.020703,0.064562
chlorophyll_a_mg_m3__lag_5,-0.020523,0.021665,-0.947292,3.435724e-01,-0.063004,0.021958
chlorophyll_a_mg_m3__lag_6,0.035088,0.021659,1.620025,1.053405e-01,-0.007381,0.077558
chlorophyll_a_mg_m3__lag_7,0.062623,0.021369,2.930509,3.411709e-03,0.020722,0.104524
chlorophyll_a_mg_m3__lag_8,0.030881,0.021322,1.448342,1.476344e-01,-0.010927,0.072689
chlorophyll_a_mg_m3__lag_9,0.026449,0.021251,1.244602,2.133831e-01,-0.015220,0.068117


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.554768,0.018995,29.205352,9.958977e-164,0.517522,0.592015
wind_speed_m_s__lag_1,0.076430,0.009802,7.797421,8.869351e-15,0.057210,0.095649
upwelling_index__lag_2,0.134434,0.018958,7.091173,1.680508e-12,0.097261,0.171607
par_umol_m2_s__lag_1,0.000944,0.000147,6.433392,1.464673e-10,0.000657,0.001232
nitrate_umol_l__lag_3,0.114758,0.019292,5.948558,3.044433e-09,0.076931,0.152586
mixed_layer_depth_m__lag_1,-0.011391,0.002769,-4.114134,3.999494e-05,-0.016820,-0.005962
chlorophyll_a_mg_m3__lag_7,0.062623,0.021369,2.930509,3.411709e-03,0.020722,0.104524
turbidity_ntu__lag_4,-0.056592,0.021647,-2.614302,8.989355e-03,-0.099038,-0.014146
sst_c__lag_5,0.044466,0.017220,2.582221,9.867495e-03,0.010701,0.078232
nitrate_umol_l__lag_13,-0.036426,0.016534,-2.203106,2.766922e-02,-0.068846,-0.004006


(LinearRegression(),
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x2c5ba2731a0>)

In [213]:
# Outliers - Moda | Lag Default | No Scaling | sig=0.05
df_outliers_mode = handle_outliers(df, blf.TARGET, threshold=6,
                            method="mode", months=12, fit_until="2023-01-01")

mode_lagged = build_and_save(df_outliers_mode, "lagged_default")
analysis_LOS(mode_lagged, scaling="standard", sig=0.05, split="2023-01-01")

# SCALING N MUDA ND PARA O OLS

chlorophyll_a_mg_m3: 4 outliers (> 6) handled with 'mode'
saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
--- Metrics (X_test) ---
R2: 0.7371568470277643
MAE: 0.18956912922814764
MSE: 0.10048556483870835
RMSE: 0.3169945817182186
MAPE: 0.11079717960500332
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.831
Model:                             OLS   Adj. R-squared:                  0.823
Method:                  Least Squares   F-statistic:                     102.3
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:29   Log-Likelihood:                 30.894
No. Observations:                 2908   AIC:                             206.2
Df Residuals:                     2774   BIC:                             1007.
Df Model:                          133            

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,1.731797,0.004546,380.979062,0.000000e+00,1.722883,1.740710
chlorophyll_a_mg_m3__lag_1,0.322728,0.011050,29.205352,9.958977e-164,0.301060,0.344395
chlorophyll_a_mg_m3__lag_2,-0.015270,0.012643,-1.207777,2.272360e-01,-0.040060,0.009521
chlorophyll_a_mg_m3__lag_3,-0.015087,0.012645,-1.193090,2.329362e-01,-0.039883,0.009708
chlorophyll_a_mg_m3__lag_4,0.012759,0.012650,1.008604,3.132524e-01,-0.012046,0.037565
chlorophyll_a_mg_m3__lag_5,-0.011942,0.012607,-0.947292,3.435724e-01,-0.036661,0.012777
chlorophyll_a_mg_m3__lag_6,0.020417,0.012603,1.620025,1.053405e-01,-0.004295,0.045129
chlorophyll_a_mg_m3__lag_7,0.036436,0.012433,2.930509,3.411709e-03,0.012057,0.060816
chlorophyll_a_mg_m3__lag_8,0.017970,0.012407,1.448342,1.476344e-01,-0.006358,0.042298
chlorophyll_a_mg_m3__lag_9,0.015392,0.012367,1.244602,2.133831e-01,-0.008857,0.039641


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,1.731797,0.004546,380.979062,0.000000e+00,1.722883,1.740710
chlorophyll_a_mg_m3__lag_1,0.322728,0.011050,29.205352,9.958977e-164,0.301060,0.344395
wind_speed_m_s__lag_1,0.103344,0.013254,7.797421,8.869351e-15,0.077356,0.129332
upwelling_index__lag_2,0.138877,0.019585,7.091173,1.680508e-12,0.100476,0.177279
par_umol_m2_s__lag_1,0.159137,0.024736,6.433392,1.464673e-10,0.110634,0.207640
nitrate_umol_l__lag_3,0.164769,0.027699,5.948558,3.044433e-09,0.110456,0.219081
mixed_layer_depth_m__lag_1,-0.075836,0.018433,-4.114134,3.999494e-05,-0.111980,-0.039692
chlorophyll_a_mg_m3__lag_7,0.036436,0.012433,2.930509,3.411709e-03,0.012057,0.060816
turbidity_ntu__lag_4,-0.022755,0.008704,-2.614302,8.989355e-03,-0.039822,-0.005688
sst_c__lag_5,0.144883,0.056108,2.582221,9.867495e-03,0.034866,0.254901


(LinearRegression(),
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x2c5ab069430>)

In [214]:
# Outliers - 12 meses | Lag Default | No Scaling | sig=0.05
df_outliers_month_12 = handle_outliers(df, blf.TARGET, threshold=6,
                            method="months_back", months=12, fit_until="2023-01-01")

month_12_lagged = build_and_save(df_outliers_month_12, "lagged_default")
analysis_LOS(month_12_lagged, scaling=None, sig=0.05, split="2023-01-01")

chlorophyll_a_mg_m3: 4 outliers (> 6) handled with 'months_back'
saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
--- Metrics (X_test) ---
R2: 0.7534018043130479
MAE: 0.18458863499410647
MSE: 0.09428626731747744
RMSE: 0.30706068995799096
MAPE: 0.10750308032820721
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.833
Model:                             OLS   Adj. R-squared:                  0.825
Method:                  Least Squares   F-statistic:                     104.1
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:29   Log-Likelihood:                 51.564
No. Observations:                 2908   AIC:                             164.9
Df Residuals:                     2774   BIC:                             965.6
Df Model:                          133    

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,2.742324,7.508096,0.365249,7.149534e-01,-11.979697,17.464346
chlorophyll_a_mg_m3__lag_1,0.561774,0.018993,29.578644,2.294041e-167,0.524533,0.599015
chlorophyll_a_mg_m3__lag_2,-0.031983,0.021807,-1.466648,1.425853e-01,-0.074743,0.010777
chlorophyll_a_mg_m3__lag_3,-0.023167,0.021811,-1.062152,2.882591e-01,-0.065935,0.019601
chlorophyll_a_mg_m3__lag_4,0.019253,0.021817,0.882509,3.775782e-01,-0.023525,0.062032
chlorophyll_a_mg_m3__lag_5,-0.018986,0.021735,-0.873527,3.824515e-01,-0.061606,0.023633
chlorophyll_a_mg_m3__lag_6,0.032866,0.021730,1.512473,1.305274e-01,-0.009743,0.075474
chlorophyll_a_mg_m3__lag_7,0.063247,0.021433,2.950992,3.194083e-03,0.021222,0.105273
chlorophyll_a_mg_m3__lag_8,0.030500,0.021378,1.426666,1.537888e-01,-0.011419,0.072419
chlorophyll_a_mg_m3__lag_9,0.023469,0.021310,1.101318,2.708538e-01,-0.018316,0.065255


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.561774,0.018993,29.578644,2.294041e-167,0.524533,0.599015
wind_speed_m_s__lag_1,0.075803,0.009732,7.789210,9.451979e-15,0.056721,0.094886
upwelling_index__lag_2,0.131782,0.018824,7.000688,3.182240e-12,0.094871,0.168693
par_umol_m2_s__lag_1,0.000950,0.000146,6.517026,8.487237e-11,0.000664,0.001236
nitrate_umol_l__lag_3,0.116124,0.019157,6.061583,1.530578e-09,0.078560,0.153688
mixed_layer_depth_m__lag_1,-0.011545,0.002748,-4.200741,2.744335e-05,-0.016934,-0.006156
chlorophyll_a_mg_m3__lag_7,0.063247,0.021433,2.950992,3.194083e-03,0.021222,0.105273
sst_c__lag_5,0.043741,0.017106,2.557124,1.060692e-02,0.010200,0.077283
turbidity_ntu__lag_4,-0.053077,0.021493,-2.469531,1.358882e-02,-0.095221,-0.010934
surface_pressure_hpa__lag_3,0.007025,0.003410,2.059769,3.951372e-02,0.000337,0.013712


(LinearRegression(),
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x2c5b9c79f70>)

In [215]:
# Outliers - 6 meses | Lag Default | No Scaling | sig=0.05
df_outliers_month_6 = handle_outliers(df, blf.TARGET, threshold=6,
                            method="months_back", months=6, fit_until="2023-01-01")

month_6_lagged = build_and_save(df_outliers_month_6, "lagged_default")
analysis_LOS(month_6_lagged, scaling=None, sig=0.05, split="2023-01-01")

chlorophyll_a_mg_m3: 4 outliers (> 6) handled with 'months_back'
saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
--- Metrics (X_test) ---
R2: 0.7379070761451187
MAE: 0.18965892987380936
MSE: 0.10018517115762668
RMSE: 0.31652041191308133
MAPE: 0.11085902362927308
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.830
Model:                             OLS   Adj. R-squared:                  0.822
Method:                  Least Squares   F-statistic:                     101.6
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:29   Log-Likelihood:                 21.975
No. Observations:                 2908   AIC:                             224.1
Df Residuals:                     2774   BIC:                             1025.
Df Model:                          133    

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,3.315248,7.584032,0.437135,6.620473e-01,-11.555669,18.186165
chlorophyll_a_mg_m3__lag_1,0.551643,0.018996,29.039951,4.001523e-162,0.514395,0.588891
chlorophyll_a_mg_m3__lag_2,-0.024598,0.021701,-1.133474,2.571134e-01,-0.067150,0.017954
chlorophyll_a_mg_m3__lag_3,-0.025936,0.021701,-1.195135,2.321365e-01,-0.068488,0.016616
chlorophyll_a_mg_m3__lag_4,0.022067,0.021711,1.016396,3.095296e-01,-0.020504,0.064638
chlorophyll_a_mg_m3__lag_5,-0.020830,0.021635,-0.962780,3.357418e-01,-0.063252,0.021592
chlorophyll_a_mg_m3__lag_6,0.036435,0.021630,1.684500,9.219752e-02,-0.005977,0.078847
chlorophyll_a_mg_m3__lag_7,0.061595,0.021345,2.885668,3.936002e-03,0.019741,0.103448
chlorophyll_a_mg_m3__lag_8,0.031885,0.021301,1.496874,1.345399e-01,-0.009883,0.073653
chlorophyll_a_mg_m3__lag_9,0.026784,0.021229,1.261662,2.071766e-01,-0.014843,0.068411


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.551643,0.018996,29.039951,4.001523e-162,0.514395,0.588891
wind_speed_m_s__lag_1,0.076396,0.009832,7.770109,1.095702e-14,0.057117,0.095675
upwelling_index__lag_2,0.134868,0.019016,7.092274,1.667419e-12,0.097581,0.172155
par_umol_m2_s__lag_1,0.000940,0.000147,6.385848,1.991539e-10,0.000652,0.001229
nitrate_umol_l__lag_3,0.114307,0.019351,5.907180,3.904286e-09,0.076364,0.152250
mixed_layer_depth_m__lag_1,-0.011343,0.002777,-4.083852,4.554871e-05,-0.016789,-0.005897
chlorophyll_a_mg_m3__lag_7,0.061595,0.021345,2.885668,3.936002e-03,0.019741,0.103448
turbidity_ntu__lag_4,-0.057173,0.021714,-2.633012,8.510101e-03,-0.099749,-0.014596
sst_c__lag_5,0.044622,0.017271,2.583675,9.826084e-03,0.010757,0.078487
nitrate_umol_l__lag_13,-0.038000,0.016584,-2.291370,2.201667e-02,-0.070518,-0.005482


(LinearRegression(),
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x2c5832cff50>)

In [216]:
# Grid Search

LOWER_IS_BETTER = {"mae", "mse", "rmse", "mape"}   # r2 is the only one where higher is better


def calc_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"r2": r2_score(y_true, y_pred),
            "mae": mean_absolute_error(y_true, y_pred),
            "mse": mse,
            "rmse": np.sqrt(mse),
            "mape": mean_absolute_percentage_error(y_true, y_pred)}









def grid_search(raw, model_cls, model_grid=None, fixed_params=None, *,
                score="r2", split, val_split=None, cv_years=None, cv_scheme="blocked",
                lag_spec=None, col=None,
                outlier_methods=("remove", "neighbors", "mean", "median", "mode",
                                 "months_back", "years_median"),
                thresholds=(4.5, 6, 8), months_list=(6, 12),
                scalings=(None, "standard", "minmax", "robust")):
    """Try every combination of outlier handling x scaling x model hyperparameters.

    split     : test period. A date, or (start, end) / (start, end, 'past'), see make_masks
    val_split : (date splits only) train < val_split <= validation < split. Rank on validation.
    cv_years  : if set (1 or 3), rank by cross-validation on the TRAIN part, using folds of
                `cv_years` years. The test period is only reported.
    cv_scheme : 'blocked' (train on all other years) or 'forward' (train only on earlier years)
    Ranking uses the sel_ columns; test_ columns are always the test period.
    """
    col = col or blf.TARGET
    PENALIZED = {"Lasso", "Ridge", "ElasticNet", "LassoCV", "RidgeCV", "ElasticNetCV"}
    if model_cls.__name__ in PENALIZED:
        if any(not s for s in scalings):
            print(f"{model_cls.__name__} is penalized, so 'no scaling' was removed from the grid.")
        scalings = tuple(s for s in scalings if s)
        if not scalings:
            raise ValueError("penalized models need at least one scaling option")
    model_grid, fixed_params = model_grid or {}, fixed_params or {}

    if isinstance(split, (tuple, list)):
        if val_split:
            raise ValueError("val_split only works with a date split; use cv_years instead")
    elif val_split is not None and pd.Timestamp(val_split) >= pd.Timestamp(split):
        val_split = None
    if val_split and cv_years:
        raise ValueError("use val_split or cv_years, not both")
    fit_ref = val_split or split          # outlier statistics come from the training rows only

    outlier_opts = [(None, None, None)]
    for m, t in product(outlier_methods, thresholds):
        for mo in (months_list if m == "months_back" else [None]):
            outlier_opts.append((m, t, mo))

    keys = list(model_grid)
    combos = list(product(*model_grid.values()))
    total = len(outlier_opts) * len(scalings) * len(combos)
    print(f"{total} setups" + (f", each fitted on every CV fold ({cv_years}-year folds, {cv_scheme})." if cv_years else "."))

    rows = []
    with redirect_stdout(io.StringIO()):
        for m, t, mo in outlier_opts:
            clean = raw if m is None else handle_outliers(
                raw, col, t, m, months=mo or 12, fit_until=fit_ref)
            lagged = blf.build_lagged_frame(clean, lag_spec=lag_spec).dropna().reset_index(drop=True)
            dates = lagged[blf.DATE]
            feats = [c for c in lagged.columns if c not in (blf.DATE, blf.TARGET)]
            X, y = lagged[feats], lagged[blf.TARGET]

            train_m, test_m = make_masks(dates, split)
            if val_split:
                fit_m = dates < pd.Timestamp(val_split)
                sel_m = ~fit_m & train_m
            else:
                fit_m, sel_m = train_m, test_m
            folds = year_folds(dates[fit_m], cv_years, cv_scheme) if cv_years else None

            for sc in scalings:
                Xf, Xs, Xt = scale_sets(sc, X[fit_m], X[sel_m], X[test_m])
                yf, ys, yt = y[fit_m], y[sel_m], y[test_m]
                fold_data = None
                if folds:   # scaler fitted inside each fold, on that fold's training years only
                    fold_data = []
                    for tr, va in folds:
                        Xa, Xb = scale_sets(sc, X.loc[tr], X.loc[va])
                        fold_data.append((Xa, Xb, y.loc[tr], y.loc[va]))

                for vals in combos:
                    params = dict(zip(keys, vals))
                    model = model_cls(**params, **fixed_params).fit(Xf, yf)
                    if fold_data:
                        fm = pd.DataFrame([calc_metrics(yb, model_cls(**params, **fixed_params)
                                                        .fit(Xa, ya).predict(Xb))
                                           for Xa, Xb, ya, yb in fold_data])
                        sel = fm.mean().to_dict()
                    else:
                        sel = calc_metrics(ys, model.predict(Xs))
                    row = {"outliers": m or "none", "threshold": t, "months": mo,
                           "scaling": sc or "none", **params}
                    row.update({f"sel_{k}": v for k, v in sel.items()})
                    row.update({f"test_{k}": v for k, v in calc_metrics(yt, model.predict(Xt)).items()})
                    rows.append(row)

    results = (pd.DataFrame(rows)
               .sort_values(f"sel_{score}", ascending=score in LOWER_IS_BETTER)
               .reset_index(drop=True))
    how = "cross-validation" if cv_years else ("validation" if val_split else "test")
    if how == "test":
        print("Warning: ranking uses the test set (optimistic).")
    print(f"Best by {how} {score}:")
    display(results.head(1).T)
    return results









'''

def grid_search(raw, model_cls, model_grid=None, fixed_params=None, *,
                score="r2", split, val_split=None, lag_spec=None, col=None,
                outlier_methods=("remove", "neighbors", "mean", "median", "mode", "months_back"),
                thresholds=(4.5, 6, 8), months_list=(6, 12),
                scalings=(None, "standard", "minmax", "robust")):
    """Try every combination of outlier handling x scaling x model hyperparameters.

    model_cls   : LinearRegression, Lasso, Ridge, ...
    model_grid  : hyperparameters to vary, e.g. {"alpha": [0.001, 0.01, 0.1]}
    fixed_params: hyperparameters kept constant, e.g. {"max_iter": 10000}
    score       : metric used to rank ('r2', 'mae', 'mse', 'rmse', 'mape')
    split       : start of the test period
    val_split   : if given, train < val_split <= validation < split <= test.
                  Ranking uses the validation period; test metrics are only reported.
    """
    col = col or blf.TARGET
    model_grid, fixed_params = model_grid or {}, fixed_params or {}
    train_end = val_split or split

    # outlier options (months only matters for 'months_back'); first one = no treatment
    outlier_opts = [(None, None, None)]
    for m, t in product(outlier_methods, thresholds):
        for mo in (months_list if m == "months_back" else [None]):
            outlier_opts.append((m, t, mo))

    keys = list(model_grid)
    combos = list(product(*model_grid.values()))
    total = len(outlier_opts) * len(scalings) * len(combos)
    print(f"Fitting {total} models...")

    rows = []
    with redirect_stdout(io.StringIO()):          # silence the helper prints
        for m, t, mo in outlier_opts:
            clean = raw if m is None else handle_outliers(
                raw, col, t, m, months=mo or 12, fit_until=train_end)
            lagged = blf.build_lagged_frame(clean, lag_spec=lag_spec)

            for sc in scalings:
                X_tr, X_rest, y_tr, y_rest = prepare_data(lagged, sc, train_end)
                if val_split:
                    is_val = lagged.loc[X_rest.index, blf.DATE] < pd.Timestamp(split)
                    X_sel, y_sel = X_rest[is_val], y_rest[is_val]
                    X_te, y_te = X_rest[~is_val], y_rest[~is_val]
                else:
                    X_sel, y_sel, X_te, y_te = X_rest, y_rest, X_rest, y_rest

                for vals in combos:
                    params = dict(zip(keys, vals))
                    model = model_cls(**params, **fixed_params).fit(X_tr, y_tr)
                    row = {"outliers": m or "none", "threshold": t, "months": mo,
                           "scaling": sc or "none", **params}
                    row.update({f"sel_{k}": v for k, v in calc_metrics(y_sel, model.predict(X_sel)).items()})
                    row.update({f"test_{k}": v for k, v in calc_metrics(y_te, model.predict(X_te)).items()})
                    rows.append(row)

    results = (pd.DataFrame(rows)
               .sort_values(f"sel_{score}", ascending=score in LOWER_IS_BETTER)
               .reset_index(drop=True))
    if not val_split:
        print("Warning: no val_split, so the ranking uses the test set (optimistic).")
    print(f"Best by {'validation' if val_split else 'test'} {score}:")
    display(results.head(1).T)
    return results
'''

'\n\ndef grid_search(raw, model_cls, model_grid=None, fixed_params=None, *,\n                score="r2", split, val_split=None, lag_spec=None, col=None,\n                outlier_methods=("remove", "neighbors", "mean", "median", "mode", "months_back"),\n                thresholds=(4.5, 6, 8), months_list=(6, 12),\n                scalings=(None, "standard", "minmax", "robust")):\n    """Try every combination of outlier handling x scaling x model hyperparameters.\n\n    model_cls   : LinearRegression, Lasso, Ridge, ...\n    model_grid  : hyperparameters to vary, e.g. {"alpha": [0.001, 0.01, 0.1]}\n    fixed_params: hyperparameters kept constant, e.g. {"max_iter": 10000}\n    score       : metric used to rank (\'r2\', \'mae\', \'mse\', \'rmse\', \'mape\')\n    split       : start of the test period\n    val_split   : if given, train < val_split <= validation < split <= test.\n                  Ranking uses the validation period; test metrics are only reported.\n    """\n    col = col or b

In [217]:
raw = blf.read_table(DATA_FILE)

# Linear regression (no hyperparameters)
res_ols = grid_search(raw, LinearRegression, score="r2",
                      split="2023-01-01")

100 setups.
Best by test r2:


,0
outliers,neighbors
threshold,4.5
months,NaN
scaling,none
sel_r2,0.851012
sel_mae,0.168349
sel_mse,0.051946
sel_rmse,0.227917
sel_mape,0.102427
test_r2,0.851012


In [218]:
# Outliers - neighbors | Lag Default | No Scaling | sig=0.05
df_outliers_neighbors = handle_outliers(df, blf.TARGET, threshold=4.5,
                            method="neighbors", fit_until="2023-01-01")

neighbors_lagged = build_and_save(df_outliers_neighbors, "lagged_default")
analysis_LOS(neighbors_lagged, scaling=None, sig=0.05, split="2023-01-01")

chlorophyll_a_mg_m3: 6 outliers (> 4.5) handled with 'neighbors'
saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
--- Metrics (X_test) ---
R2: 0.8510119089118575
MAE: 0.16834850368171747
MSE: 0.051946083812783456
RMSE: 0.22791683529915788
MAPE: 0.10242660151449495
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.837
Model:                             OLS   Adj. R-squared:                  0.829
Method:                  Least Squares   F-statistic:                     106.9
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:34   Log-Likelihood:                 91.881
No. Observations:                 2908   AIC:                             84.24
Df Residuals:                     2774   BIC:                             884.9
Df Model:                          133   

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,3.073944,7.404381,0.415152,6.780627e-01,-11.444712,17.592599
chlorophyll_a_mg_m3__lag_1,0.564447,0.018991,29.721835,9.096757e-169,0.527209,0.601685
chlorophyll_a_mg_m3__lag_2,-0.036418,0.021837,-1.667764,9.547555e-02,-0.079236,0.006399
chlorophyll_a_mg_m3__lag_3,-0.015720,0.021849,-0.719465,4.719150e-01,-0.058562,0.027123
chlorophyll_a_mg_m3__lag_4,0.007808,0.021862,0.357147,7.210086e-01,-0.035059,0.050674
chlorophyll_a_mg_m3__lag_5,-0.018272,0.021786,-0.838718,4.017000e-01,-0.060990,0.024446
chlorophyll_a_mg_m3__lag_6,0.037297,0.021792,1.711525,8.709595e-02,-0.005433,0.080026
chlorophyll_a_mg_m3__lag_7,0.055113,0.021486,2.565030,1.036883e-02,0.012982,0.097243
chlorophyll_a_mg_m3__lag_8,0.041727,0.021428,1.947258,5.160461e-02,-0.000291,0.083744
chlorophyll_a_mg_m3__lag_9,0.017404,0.021367,0.814546,4.154021e-01,-0.024492,0.059301


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.564447,0.018991,29.721835,9.096757e-169,0.527209,0.601685
wind_speed_m_s__lag_1,0.075449,0.009600,7.859517,5.471090e-15,0.056625,0.094272
upwelling_index__lag_2,0.129944,0.018570,6.997682,3.250032e-12,0.093533,0.166356
par_umol_m2_s__lag_1,0.000944,0.000144,6.564730,6.199084e-11,0.000662,0.001226
nitrate_umol_l__lag_3,0.118903,0.018895,6.292724,3.613397e-10,0.081853,0.155953
mixed_layer_depth_m__lag_1,-0.011725,0.002711,-4.325167,1.577970e-05,-0.017041,-0.006410
sst_c__lag_5,0.046698,0.016873,2.767612,5.684346e-03,0.013613,0.079783
chlorophyll_a_mg_m3__lag_7,0.055113,0.021486,2.565030,1.036883e-02,0.012982,0.097243
turbidity_ntu__lag_4,-0.049436,0.021192,-2.332719,1.973406e-02,-0.090991,-0.007881
salinity_psu__lag_4,-0.182223,0.083460,-2.183343,2.909406e-02,-0.345873,-0.018572


(LinearRegression(),
 <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x2c5ab249df0>)

In [219]:
def prune_lags(clean, sig, split, scaling=None, lag_spec=None,
               max_rounds=1, remove="non_significant"):
    """Remove lag columns by p-value and return the reduced lag_spec.

    remove: 'non_significant' -> drop lags with p >= sig (normal use)
            'significant'     -> drop lags with p <  sig (the literal opposite)
    max_rounds: 1 = filter once; >1 = refit and filter again until nothing changes.
    """
    spec = lag_spec or blf.DEFAULT_LAGS
    for r in range(max_rounds):
        lagged = blf.build_lagged_frame(clean, lag_spec=spec)
        X_tr, _, y_tr, _ = prepare_data(lagged, scaling, split)
        pv = sm.OLS(y_tr, sm.add_constant(X_tr)).fit().pvalues.drop("const")
        pv = pv[pv.index.str.contains("__lag_")]          # ignore annual_sin / annual_cos

        drop = pv[pv >= sig] if remove == "non_significant" else pv[pv < sig]
        if drop.empty:
            break
        kept = pv.index.difference(drop.index)
        if len(kept) == 0:
            raise ValueError(f"sig={sig}: no lag columns left")

        spec = {}
        for name in kept:
            col, lag = name.rsplit("__lag_", 1)
            spec.setdefault(col, []).append(int(lag))
        spec = {c: sorted(l) for c, l in spec.items()}
    return spec

In [220]:
split = "2023-01-01"
clean = handle_outliers(df, blf.TARGET, threshold=4.5, method="neighbors", fit_until=split)

for sig in [0.01, 0.05, 0.10]:
    spec = prune_lags(clean, sig, split, scaling=None, max_rounds=1)
    n = sum(len(v) for v in spec.values())
    print(f"\n===== sig={sig}: {n} lag columns kept =====")
    reduced = build_and_save(clean, f"lagged_best_sig{sig}", lag_spec=spec)
    analysis_LOS(reduced, scaling=None, sig=sig, split=split)

chlorophyll_a_mg_m3: 6 outliers (> 4.5) handled with 'neighbors'

===== sig=0.01: 7 lag columns kept =====
saved (3282, 11) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_best_sig0.01.csv
--- Metrics (X_test) ---
R2: 0.8443690506058693
MAE: 0.17398446174962956
MSE: 0.05426217815159318
RMSE: 0.23294243527445396
MAPE: 0.1052753656311729
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.819
Model:                             OLS   Adj. R-squared:                  0.818
Method:                  Least Squares   F-statistic:                     1460.
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:34   Log-Likelihood:                -59.229
No. Observations:                 2917   AIC:                             138.5
Df Residuals:                     2907   BIC:                             198

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,-0.050944,0.146221,-0.348405,7.275615e-01,-0.337651,0.235763
chlorophyll_a_mg_m3__lag_1,0.537359,0.012344,43.531836,0.000000e+00,0.513155,0.561563
mixed_layer_depth_m__lag_1,-0.013601,0.002650,-5.132082,3.054159e-07,-0.018797,-0.008404
nitrate_umol_l__lag_3,0.115574,0.007770,14.874499,2.754211e-48,0.100339,0.130810
par_umol_m2_s__lag_1,0.000523,0.000118,4.452087,8.824273e-06,0.000293,0.000754
sst_c__lag_5,0.006140,0.008722,0.703938,4.815276e-01,-0.010962,0.023241
upwelling_index__lag_2,0.128697,0.006140,20.960228,5.841421e-91,0.116658,0.140737
wind_speed_m_s__lag_1,0.070301,0.005919,11.876294,8.442862e-32,0.058694,0.081908
annual_sin,0.026113,0.016599,1.573183,1.157853e-01,-0.006434,0.058659
annual_cos,0.178231,0.046180,3.859489,1.160958e-04,0.087682,0.268780


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.537359,0.012344,43.531836,0.000000e+00,0.513155,0.561563
upwelling_index__lag_2,0.128697,0.006140,20.960228,5.841421e-91,0.116658,0.140737
nitrate_umol_l__lag_3,0.115574,0.007770,14.874499,2.754211e-48,0.100339,0.130810
wind_speed_m_s__lag_1,0.070301,0.005919,11.876294,8.442862e-32,0.058694,0.081908
mixed_layer_depth_m__lag_1,-0.013601,0.002650,-5.132082,3.054159e-07,-0.018797,-0.008404
par_umol_m2_s__lag_1,0.000523,0.000118,4.452087,8.824273e-06,0.000293,0.000754
annual_cos,0.178231,0.046180,3.859489,1.160958e-04,0.087682,0.268780



===== sig=0.05: 13 lag columns kept =====
saved (3275, 17) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_best_sig0.05.csv
--- Metrics (X_test) ---
R2: 0.8544743611199963
MAE: 0.16515930377082014
MSE: 0.05073886764343658
RMSE: 0.22525289708111765
MAPE: 0.10074622113789489
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.827
Model:                             OLS   Adj. R-squared:                  0.827
Method:                  Least Squares   F-statistic:                     925.0
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:34   Log-Likelihood:                 11.478
No. Observations:                 2910   AIC:                             9.043
Df Residuals:                     2894   BIC:                             104.7
Df Model:                           15                      

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,2.929020,1.089776,2.687726,7.235221e-03,0.792204,5.065835
chlorophyll_a_mg_m3__lag_1,0.512053,0.012435,41.179782,3.659390e-292,0.487672,0.536435
chlorophyll_a_mg_m3__lag_7,0.094337,0.010242,9.210468,6.061232e-20,0.074254,0.114420
chlorophyll_a_mg_m3__lag_12,0.020191,0.009959,2.027481,4.270476e-02,0.000664,0.039718
cloud_fraction__lag_0,-0.010006,0.043763,-0.228631,8.191722e-01,-0.095815,0.075804
current_speed_m_s__lag_7,-0.199856,0.062065,-3.220080,1.295731e-03,-0.321552,-0.078159
mixed_layer_depth_m__lag_1,-0.013064,0.002596,-5.033223,5.118240e-07,-0.018153,-0.007975
nitrate_umol_l__lag_3,0.119523,0.008015,14.912603,1.650327e-48,0.103807,0.135238
par_umol_m2_s__lag_1,0.000573,0.000121,4.722038,2.446077e-06,0.000335,0.000810
salinity_psu__lag_4,-0.092394,0.031063,-2.974411,2.959780e-03,-0.153302,-0.031486


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.512053,0.012435,41.179782,3.659390e-292,0.487672,0.536435
upwelling_index__lag_2,0.134092,0.006069,22.096091,4.174872e-100,0.122193,0.145991
nitrate_umol_l__lag_3,0.119523,0.008015,14.912603,1.650327e-48,0.103807,0.135238
wind_speed_m_s__lag_1,0.070440,0.005792,12.160960,3.197460e-33,0.059082,0.081797
chlorophyll_a_mg_m3__lag_7,0.094337,0.010242,9.210468,6.061232e-20,0.074254,0.114420
mixed_layer_depth_m__lag_1,-0.013064,0.002596,-5.033223,5.118240e-07,-0.018153,-0.007975
par_umol_m2_s__lag_1,0.000573,0.000121,4.722038,2.446077e-06,0.000335,0.000810
annual_cos,0.202400,0.051375,3.939634,8.353420e-05,0.101664,0.303137
current_speed_m_s__lag_7,-0.199856,0.062065,-3.220080,1.295731e-03,-0.321552,-0.078159
salinity_psu__lag_4,-0.092394,0.031063,-2.974411,2.959780e-03,-0.153302,-0.031486



===== sig=0.1: 22 lag columns kept =====
saved (3274, 26) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_best_sig0.1.csv
--- Metrics (X_test) ---
R2: 0.8579569654113797
MAE: 0.16351916036929323
MSE: 0.04952462526281616
RMSE: 0.22254128889447944
MAPE: 0.09952318524741702
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.832
Model:                             OLS   Adj. R-squared:                  0.830
Method:                  Least Squares   F-statistic:                     593.7
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         18:40:35   Log-Likelihood:                 47.770
No. Observations:                 2909   AIC:                            -45.54
Df Residuals:                     2884   BIC:                             103.8
Df Model:                           24                        

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
const,2.580613,1.944386,1.327212,1.845435e-01,-1.231914,6.393140
chlorophyll_a_mg_m3__lag_1,0.565429,0.017790,31.784086,2.454693e-190,0.530548,0.600311
chlorophyll_a_mg_m3__lag_2,-0.066964,0.015681,-4.270289,2.015266e-05,-0.097712,-0.036216
chlorophyll_a_mg_m3__lag_6,0.032428,0.015758,2.057950,3.968466e-02,0.001531,0.063325
chlorophyll_a_mg_m3__lag_7,0.054374,0.020204,2.691315,7.158171e-03,0.014759,0.093989
chlorophyll_a_mg_m3__lag_8,0.051061,0.015945,3.202306,1.378154e-03,0.019796,0.082325
chlorophyll_a_mg_m3__lag_12,0.030132,0.010647,2.830202,4.684052e-03,0.009256,0.051008
cloud_fraction__lag_0,0.038832,0.044406,0.874465,3.819377e-01,-0.048240,0.125903
current_speed_m_s__lag_7,-0.181777,0.065362,-2.781072,5.453176e-03,-0.309939,-0.053616
mixed_layer_depth_m__lag_1,-0.012755,0.002572,-4.959732,7.468176e-07,-0.017798,-0.007713


,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
chlorophyll_a_mg_m3__lag_1,0.565429,0.017790,31.784086,2.454693e-190,0.530548,0.600311
upwelling_index__lag_2,0.129704,0.006048,21.444674,8.842131e-95,0.117844,0.141563
nitrate_umol_l__lag_3,0.113996,0.008068,14.129852,6.997084e-44,0.098177,0.129815
wind_speed_m_s__lag_1,0.070590,0.005742,12.294418,6.749494e-34,0.059332,0.081848
par_umol_m2_s__lag_1,0.000771,0.000128,6.040173,1.736874e-09,0.000521,0.001021
mixed_layer_depth_m__lag_1,-0.012755,0.002572,-4.959732,7.468176e-07,-0.017798,-0.007713
sst_c__lag_2,-0.047986,0.010187,-4.710698,2.585521e-06,-0.067960,-0.028012
chlorophyll_a_mg_m3__lag_2,-0.066964,0.015681,-4.270289,2.015266e-05,-0.097712,-0.036216
nitrate_umol_l__lag_13,-0.026196,0.006694,-3.913461,9.308593e-05,-0.039321,-0.013071
chlorophyll_a_mg_m3__lag_8,0.051061,0.015945,3.202306,1.378154e-03,0.019796,0.082325


In [221]:
''' LAGS A MAO E AO CONTRARIO
# custom lags
my_lags = make_lags(
    chlorophyll_a_mg_m3=[i for i in range(1, 15) if i not in [1, 7, 12]],
    sst_c=[i for i in range(0, 15) if i != 5],
    par_umol_m2_s=[i for i in range(0, 8) if i != 1],
    nitrate_umol_l=[i for i in range(0, 15) if i != 3],
    wind_speed_m_s=[i for i in range(0, 8) if i != 1],
    upwelling_index=[i for i in range(0, 15) if i != 2],
    mixed_layer_depth_m=[i for i in range(0, 8) if i != 1],
    salinity_psu=[i for i in range(0, 8) if i != 4],
    current_speed_m_s=[i for i in range(0, 8) if i != 7],
    river_discharge_index=[i for i in range(0, 8)],
    cloud_fraction=[i for i in range(0, 8) if i != 0],
    surface_pressure_hpa=[i for i in range(0, 8)],
    turbidity_ntu=[i for i in range(0, 8) if i != 4],
)

lagged_short = build_and_save(raw, "lagged_short", lag_spec=my_lags)



# Outliers - neighbors | Lag Default | No Scaling | sig=0.05
df_outliers_neighbors = handle_outliers(df, blf.TARGET, threshold=4.5,
                            method="neighbors", fit_until="2023-01-01")

neighbors_lagged = build_and_save(df_outliers_neighbors, "lagged_short")
_, ols_full=analysis_LOS(neighbors_lagged, scaling=None, sig=0.05, split="2023-01-01")
'''

' LAGS A MAO E AO CONTRARIO\n# custom lags\nmy_lags = make_lags(\n    chlorophyll_a_mg_m3=[i for i in range(1, 15) if i not in [1, 7, 12]],\n    sst_c=[i for i in range(0, 15) if i != 5],\n    par_umol_m2_s=[i for i in range(0, 8) if i != 1],\n    nitrate_umol_l=[i for i in range(0, 15) if i != 3],\n    wind_speed_m_s=[i for i in range(0, 8) if i != 1],\n    upwelling_index=[i for i in range(0, 15) if i != 2],\n    mixed_layer_depth_m=[i for i in range(0, 8) if i != 1],\n    salinity_psu=[i for i in range(0, 8) if i != 4],\n    current_speed_m_s=[i for i in range(0, 8) if i != 7],\n    river_discharge_index=[i for i in range(0, 8)],\n    cloud_fraction=[i for i in range(0, 8) if i != 0],\n    surface_pressure_hpa=[i for i in range(0, 8)],\n    turbidity_ntu=[i for i in range(0, 8) if i != 4],\n)\n\nlagged_short = build_and_save(raw, "lagged_short", lag_spec=my_lags)\n\n\n\n# Outliers - neighbors | Lag Default | No Scaling | sig=0.05\ndf_outliers_neighbors = handle_outliers(df, blf.TARG

In [222]:
# Lasso with yearly CV (9 folds) or 3-year CV (3 folds), test = 2023
res_l1 = grid_search(raw, Lasso, {"alpha": [0.001, 0.01, 0.1]}, {"max_iter": 10000},
                     split="2023-01-01", cv_years=1)
res_l3 = grid_search(raw, Lasso, {"alpha": [0.001, 0.01, 0.1]}, {"max_iter": 10000},
                     split="2023-01-01", cv_years=3, cv_scheme="forward")

# Linear regression with the winning setup on different test periods
compare_splits(raw, LinearRegression,
    splits={"test 2023 (train <2023)":       "2023-01-01",
            "test 2022-23 (train <2022)":    ("2022-01-01", "2024-01-01"),
            "test 2022 (train all others)":  ("2022-01-01", "2023-01-01"),
            "test 2022 (train <2022 only)":  ("2022-01-01", "2023-01-01", "past")},
    scaling=None,
    outlier=dict(threshold=6, method="months_back", months=6))


res_l1 = grid_search(raw, Lasso, {"alpha": [0.001, 0.01, 0.1]}, {"max_iter": 10000},
                     split="2023-01-01", cv_years=1,
                     scalings=("standard", "minmax", "robust"))

Lasso is penalized, so 'no scaling' was removed from the grid.
225 setups, each fitted on every CV fold (1-year folds, blocked).
Best by cross-validation r2:


,0
outliers,neighbors
threshold,4.5
months,NaN
scaling,standard
alpha,0.001
sel_r2,0.821157
sel_mae,0.174931
sel_mse,0.059689
sel_rmse,0.242886
sel_mape,0.10659


Lasso is penalized, so 'no scaling' was removed from the grid.
225 setups, each fitted on every CV fold (3-year folds, forward).
Best by cross-validation r2:


,0
outliers,neighbors
threshold,4.5
months,NaN
scaling,robust
alpha,0.001
sel_r2,0.830495
sel_mae,0.172803
sel_mse,0.056065
sel_rmse,0.236614
sel_mape,0.109054


225 setups, each fitted on every CV fold (1-year folds, blocked).
Best by cross-validation r2:


,0
outliers,neighbors
threshold,4.5
months,NaN
scaling,standard
alpha,0.001
sel_r2,0.821157
sel_mae,0.174931
sel_mse,0.059689
sel_rmse,0.242886
sel_mape,0.10659
